<table>
  <tr>
    <td><div align="left"><font size="30">Block-diagram simulation</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

<table align="center" style="border: none; border-collapse: collapse;">
  <tr>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">🧠</td>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">Block diagram thinking</td>
  </tr>
  <tr>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">🐍</td>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">Python coding</td>
  </tr>
</table>

(c) Peter Corke 2026

Presented at ICRA 2026 as part of Keynote Tutorial 3 (the "lost tutorial"). It needs `roboticstoolbox-python` as well as `bdsim`.

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(packages=['roboticstoolbox-python'])


In [ ]:
import bdsim
import matplotlib.pyplot as plt

# A first-order system




Here's a hand drawn sketch of a simple first-order system with proportional feedback

![Block diagram sketch](https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bd1-sketch.png)

We model it using `bdsim` by writing Python code to represent the model.  Python means you can use your
favourite IDE, debugger, and version control.

In [ ]:
sim = bdsim.BDSim(animation=True)  # create simulator
bd = sim.blockdiagram()  # create an empty block diagram

# define the blocks
demand = bd.STEP(T=1, name="demand")
sum = bd.SUM("+-")
gain = bd.GAIN(10)
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

# connect the blocks
bd.connect(demand, sum[0], scope[1])
bd.connect(plant, sum[1])
bd.connect(sum, gain)
bd.connect(gain, plant)
bd.connect(plant, scope[0])

Before we can run the code, we need to compile it. That does a bunch of checks and builds the datastructures required for execution.

In [ ]:
bd.compile(verbose=True)  # check the diagram

and this is a succinct representation of the model, showing each block and where it's inputs are sourced from.

In [ ]:

bd.report_summary()

Now we can run the model.  In this case for 5 seconds, and we'll see the output of the `SCOPE` block.  We will also record the outputs of the `demand` and `sum` blocks.

In [ ]:
out = sim.run(bd, T=5, watch=[demand, sum])  # simulate for 5s

In [ ]:
print(out)

In [ ]:
from IPython.display import Markdown, display

mermaid = bd.graph(format="mermaid") # export a graph in Mermaid format
display(Markdown(f"``` mermaid\n{mermaid}\n```")) # Render dynamically via Jupyter's built-in Mermaid engine

# The "bouncing ball" problem



This is a classic hybrid simulation problem with continuous dynamics (the ball flight) and discrete dynamics (the impact). 

In [ ]:
import bdsim

# define constants
g = -9.81  # gravity m/s2
e = 0.8  # coefficient of restitution
h0 = 10  # initial height m

sim = bdsim.BDSim(animation=True)  # create simulator
bd = sim.blockdiagram()  # create an empty block diagram

# define the blocks
gravity = bd.CONSTANT(g, name="gravity")

# chain of integrators for ball flight dynamics
velocity = bd.INTEGRATOR(name="velocity", x0=0)  # initial height is initial velocity
position = bd.INTEGRATOR(name="position", x0=h0)

scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

# ---------------- the impact event detection and response -------------------- #
# detect when the ball hits the ground (position=0) and trigger an event
def impact(event_block, state_map):
    state_map[velocity] *= -e  # reverse velocity and apply restitution

# connect the impact function to the event triggered when position crosses zero from above
ground = bd.EVENT("-", impact)

# ----------------------------------------------------------------------------- #

# connect the blocks together
bd.connect(gravity, velocity)
bd.connect(velocity, position)
bd.connect(velocity, scope[0])
bd.connect(position, scope[1])
bd.connect(position, ground)

bd.compile()  # check the diagram
bd.report_summary()

In [ ]:
out = sim.run(bd, T=5)  # , watch=[demand, sum])  # simulate for 5s

# Cart-pole system



This is another classical simulation problem, a non-linear system comprising two coupled dynamical systems.

In [ ]:
%run support/cartpole.py -q 

# Quadrotor flight controller


This block diagram is a complete controller for a quadrotor and drives it to follow a circular path.  The controller has nested loops, from the inside going outwards:

* roll/pitch attitude control
* CoM velocity control
* CoM position control

and a separate yaw controller, independent of the CoM motion

![quadrotor block diagram model](https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/fig4_24.png)

You can open the file `quadrotor.py` to see the model, it's quite large.

In [ ]:
%run support/quadrotor.py -q --tiles 2x2

# The block diagram editor



The nice looking block diagrams above were drawn with `bdedit` which is a Qt-based editor included with `bdsim`.  It's run from the command-line, and can't be run in a notebook. It provides a fairly complete block diagram drawing environment and the results are saved in JSON files with a `.bd` extension.

![complex](https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/computed-torque.png)



`.bd` files can be parsed to build a `bdim` block diagram which can then be simulated. For example

In [ ]:
from bdsim import BDSim, bdload
from roboticstoolbox.models.DH import Puma560


sim = BDSim(animation=True, quiet=True)
bd = sim.blockdiagram()

# create objects the loaded model will need
robot = Puma560().nofriction()
clock_fb = bd.clock(50, unit="Hz")

# load the model
bd = bdload(bd, "support/computed-torque.bd", globalvars=globals())

# compile and run
bd.compile()
sim.report(bd, "summary")
out = sim.run(bd, 5)
print(out)


# What next for `bdsim`?

Not yet releasable but these things are under development:

* web-based editor to replace Qt-based `bdedit`
* soft real-time version, running on RaspberryPi and host + Arduino/Firmata
* C++ code generation